In [1]:
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,classification_report

In [2]:
df=pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df=df.drop("customerID",axis=1)

In [5]:
df=df.drop("PaymentMethod",axis=1)

In [6]:
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [7]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["gender"]=le.fit_transform(df["gender"])
df["Partner"]=le.fit_transform(df["Partner"])
df["Dependents"]=le.fit_transform(df["Dependents"])
df["PhoneService"]=le.fit_transform(df["PhoneService"])

In [8]:
df["InternetService"]=le.fit_transform(df["InternetService"])
df["OnlineSecurity"]=le.fit_transform(df["OnlineSecurity"])
df["OnlineBackup"]=le.fit_transform(df["OnlineBackup"])
df["DeviceProtection"]=le.fit_transform(df["DeviceProtection"])

df["TechSupport"]=le.fit_transform(df["TechSupport"])
df["StreamingTV"]=le.fit_transform(df["StreamingTV"])
df["StreamingMovies"]=le.fit_transform(df["StreamingMovies"])
df["Contract"]=le.fit_transform(df["Contract"])
df["PaperlessBilling"]=le.fit_transform(df["PaperlessBilling"])
df["Churn"]=le.fit_transform(df["Churn"])

In [9]:
df["TotalCharges"]=pd.to_numeric(df["TotalCharges"],errors="coerce")
df["TotalCharges"]=df["TotalCharges"].fillna(0)

In [10]:
df["AvgMonthlyCharge"]=df["TotalCharges"]/(df["tenure"]+1)
df["IsSeniorAndContract"]=((df["SeniorCitizen"]==1)&(df["Contract"]=="Month-to-month")).astype(int)

In [14]:
x=pd.get_dummies(df.drop("Churn",axis=1),drop_first=True)
y=df["Churn"]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [22]:
model=LogisticRegression(class_weight="balanced",max_iter=2000)
model.fit(x_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [23]:
y_pred=model.predict(x_test)
from sklearn.metrics import recall_score
print("accuracy:",accuracy_score(y_test,y_pred)*100,"%")
print("precision:",precision_score(y_test,y_pred)*100,"%")
print("recall:",recall_score(y_test,y_pred)*100,"%")
print("classification_report:",classification_report(y_test,y_pred))

accuracy: 76.5081618168914 %
precision: 53.671328671328666 %
recall: 82.30563002680965 %
classification_report:               precision    recall  f1-score   support

           0       0.92      0.74      0.82      1036
           1       0.54      0.82      0.65       373

    accuracy                           0.77      1409
   macro avg       0.73      0.78      0.74      1409
weighted avg       0.82      0.77      0.78      1409



In [27]:
#XGBoost
import xgboost as xgb
xgb_clf=xgb.XGBClassifier(
    scale_pos_weight=2.8,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    colsample_bytree=0.8,
    gamma=0.1,
    eval_metric="logloss",
    random_state=42
)

In [40]:
model=xgb_clf.fit(x_train,y_train)
y_pred=model.predict_proba(x_test)[:,1]
probs=(y_pred>0.50).astype(int)
print("accuracy:",accuracy_score(y_test,probs))
print("precision:",precision_score(y_test,probs)*100,"%")
print("recall:",recall_score(y_test,probs)*100,"%")
print("classification_report:",classification_report(y_test,probs))

accuracy: 0.7601135557132718
precision: 52.93132328308208 %
recall: 84.71849865951742 %
classification_report:               precision    recall  f1-score   support

           0       0.93      0.73      0.82      1036
           1       0.53      0.85      0.65       373

    accuracy                           0.76      1409
   macro avg       0.73      0.79      0.73      1409
weighted avg       0.82      0.76      0.77      1409



In [43]:
from sklearn.metrics import roc_auc_score
print("AUC",roc_auc_score(y_test,y_pred))

AUC 0.8595119918846461
